# Embedding as a first-class primitive

An **embedding** layer is a row gather: given a table of learned vectors
`(num_embeddings, dim)` and an integer index array, return the looked-up rows. In
pycograd it is a real primitive (`ops.d_embedding`) behind the public
`pg.embedding`, so its backward **scatter-adds** the cotangent into the looked-up
rows, and it composes with the `|>` pipescript DSL, ambient `params{ ... }` weights,
`vmap`, `jvp`, `capture`, and the compile backends (where it lowers to
`F.embedding` / `jnp.take` / `tf.gather`).

This notebook trains a tiny bag-of-tokens classifier, then shows `padding_idx` and
the transform surfaces.

In [1]:
%load_ext pycograd

In [2]:
import numpy as np
import pycograd as pg

rng = np.random.default_rng(0)
VOCAB, DIM, K = 32, 8, 3        # vocabulary, embedding dim, num classes
N, L = 128, 6                   # N sequences, each L tokens long

# Each sequence is L token ids. The label is generated by a *teacher* that gives
# every token a per-class affinity and picks the class with the highest mean affinity
# -- exactly the shape a mean-pooled embedding + linear head can represent, so the
# signal is learnable.
ids = rng.integers(1, VOCAB, size=(N, L))   # reserve id 0 as a 'pad' token
teacher = rng.standard_normal((VOCAB, K))
labels = teacher[ids].mean(axis=1).argmax(axis=1)
Y = np.eye(K)[labels]                        # one-hot targets
ids.shape, Y.shape

((128, 6), (128, 3))

## A bag-of-tokens classifier with a `|>` forward

The forward is one pipeline: gather token vectors with `embedding(table, $)`
(the piped value is the token-id array, `$` is the index hole), **mean-pool** over
the sequence axis, then a linear head. Because the gather is on the tape,
`weights.grad` differentiates through it straight into `table`.

In [3]:
with params{
    table = 0.1 * rng.standard_normal((VOCAB, DIM))
    w = 0.1 * rng.standard_normal((DIM, K))
    b = np.zeros(K)
} as weights:
    forward = $ |> pg.embedding(table, $) |> np.mean($, axis=1) |> $ @ w + b
    objective = |> ids |> forward |> pg.cross_entropy($, Y)
    for step in range(300):
        value, grads = weights.grad(objective)
        weights.step(grads, 0.3)
    logits = forward(ids)

pred = np.argmax(np.asarray(getattr(logits, 'value', logits)), axis=1)
print(f'final loss {float(value):.4f}   train accuracy {(pred == labels).mean():.3f}')

final loss 0.6758   train accuracy 0.703


## `padding_idx`: hold a row fixed

A real sequence model pads short sequences with a reserved token. `padding_idx`
gives that row **zero gradient** (PyTorch semantics) so it never moves during
training -- while the forward is still a plain gather. Here id `0` is the pad token;
after a grad step its row's gradient is exactly zero.

In [4]:
with params{
    table = 0.1 * rng.standard_normal((VOCAB, DIM))
    w = 0.1 * rng.standard_normal((DIM, K))
    b = np.zeros(K)
} as weights:
    # mix some pad tokens (id 0) into the inputs
    padded = ids.copy()
    padded[:, -2:] = 0
    forward = $ |> pg.embedding(table, $, padding_idx=0) |> np.mean($, axis=1) |> $ @ w + b
    objective = |> padded |> forward |> pg.cross_entropy($, Y)
    value, grads = weights.grad(objective)

g_table = np.asarray(grads['table'])
print('pad row (id 0) grad norm  :', float(np.linalg.norm(g_table[0])))   # exactly 0
print('other rows grad norm (sum):', float(np.linalg.norm(g_table[1:])))  # nonzero

pad row (id 0) grad norm  : 0.0
other rows grad norm (sum): 0.01846738114213135


## It composes with the transforms

`embedding` is one primitive, so it rides every interpreter. A couple of quick
checks: forward-mode `jvp` (the gather is linear in the table, so the tangent gathers
identically), and graph `capture` + `value_and_grad` on the captured graph.

In [5]:
table0 = rng.standard_normal((VOCAB, DIM))
idx = np.array([[1, 3], [3, 7]])   # row 3 looked up twice

# jvp: tangent of an embedding is the embedding of the tangent
p, t = pg.jvp(lambda T: pg.embedding(T, idx), (table0,), (np.ones_like(table0),))
assert np.allclose(np.asarray(t), np.ones_like(table0)[idx])

# capture -> graph, then differentiate the graph; row 3's gradient is 2
g = pg.capture(lambda T: pg.embedding(T, idx).sum(), table0)
val, (grad_t,) = pg.value_and_grad(g)(table0)
print('graph value      :', float(np.asarray(val)))
print('grad row 3 (==2) :', np.asarray(grad_t)[3])
print('captured graph   :')
print(g.pretty())

graph value      : 1.4055052710479892
grad row 3 (==2) : [2. 2. 2. 2. 2. 2. 2. 2.]
captured graph   :
graph(%0:f64[32,8]) {
  %1 = embedding %0 {indices=array([[1, 3],
       [3, 7]]), padding_idx=None} -> f64[2,2,8]
  %2 = sum %1 -> f64[]
  outputs: %2
}


## On a compile backend

Under a delegate backend the same `embedding` symbol lowers to the framework's
native gather (`torch.nn.functional.embedding`, `jnp.take`, `tf.gather`) -- honoring
`padding_idx` -- instead of fancy-indexing. The cell below runs only if torch is
installed; the forward and the table gradient match the numpy tape.

In [6]:
try:
    import torch  # noqa: F401
    import pycograd.compile as C

    def loss(pdict):
        e = pg.embedding(pdict['table'], idx, padding_idx=3)
        return np.sum(e)

    model = {'table': table0}
    v_np, g_np = C.value_and_grad(loss, backend='numpy')(model)
    v_t,  g_t  = C.value_and_grad(loss, backend='torch')(model)
    print('value match     :', np.allclose(v_np, v_t, atol=1e-5))
    print('grad match       :', np.allclose(np.asarray(g_np['table']),
                                            np.asarray(g_t['table']), atol=1e-5))
except ImportError:
    print('torch not installed -- skipping the compile-backend check')

torch not installed -- skipping the compile-backend check
